In [0]:
%sql
select * from workspace.sliver.prod_info

prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt,category_key
210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,null,R,2003-07-01,null,CO-RF
211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,null,R,2003-07-01,null,CO-RF
212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12,S,2011-07-01,2007-12-28,AC-HE
213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14,S,2012-07-01,2008-12-27,AC-HE
214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13,S,2013-07-01,null,AC-HE
215,AC-HE-HL-U509,Sport-100 Helmet- Black,12,S,2011-07-01,2007-12-28,AC-HE
216,AC-HE-HL-U509,Sport-100 Helmet- Black,14,S,2012-07-01,2008-12-27,AC-HE
217,AC-HE-HL-U509,Sport-100 Helmet- Black,13,S,2013-07-01,null,AC-HE
218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3,M,2011-07-01,2007-12-28,CL-SO
219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3,M,2011-07-01,2007-12-28,CL-SO


In [0]:
%sql
select * from workspace.sliver.px_cat_g1v2

ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


In [0]:
query = """
select
    row_number() over (order by pr.prd_start_dt) as product_key,
    pr.prd_id           as product_id,
    pr.prd_key          as product_number,
    pr.prd_nm           as product_name,
    pr.category_key     as category_id,
    cat.CAT             as category,
    cat.SUBCAT          as subcategory,
    cat.MAINTENANCE,
    pr.prd_cost         as product_cost,
    pr.prd_line         as product_line,
    pr.prd_start_dt     as start_date,
    pr.prd_end_dt       as end_date
from workspace.sliver.prod_info as pr
left join workspace.sliver.px_cat_g1v2 as cat
    on replace(pr.category_key, '-', '_') = cat.ID
"""
df = spark.sql(query)

In [0]:
df.createOrReplaceTempView("tmp_dim_products")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_products
USING DELTA
AS
SELECT * FROM tmp_dim_products;

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from workspace.gold.dim_products

product_key,product_id,product_number,product_name,category_id,category,subcategory,MAINTENANCE,product_cost,product_line,start_date,end_date
1,210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,CO-RF,Components,Road Frames,Yes,null,R,2003-07-01,null
2,211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,CO-RF,Components,Road Frames,Yes,null,R,2003-07-01,null
3,212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,AC-HE,Accessories,Helmets,Yes,12,S,2011-07-01,2007-12-28
4,215,AC-HE-HL-U509,Sport-100 Helmet- Black,AC-HE,Accessories,Helmets,Yes,12,S,2011-07-01,2007-12-28
5,218,CL-SO-SO-B909-M,Mountain Bike Socks- M,CL-SO,Clothing,Socks,No,3,M,2011-07-01,2007-12-28
6,219,CL-SO-SO-B909-L,Mountain Bike Socks- L,CL-SO,Clothing,Socks,No,3,M,2011-07-01,2007-12-28
7,220,AC-HE-HL-U509-B,Sport-100 Helmet- Blue,AC-HE,Accessories,Helmets,Yes,12,S,2011-07-01,2007-12-28
8,223,CL-CA-CA-1098,AWC Logo Cap,CL-CA,Clothing,Caps,No,6,S,2011-07-01,2007-12-28
9,226,CL-JE-LJ-0192-S,Long-Sleeve Logo Jersey- S,CL-JE,Clothing,Jerseys,No,32,S,2011-07-01,2007-12-28
10,229,CL-JE-LJ-0192-M,Long-Sleeve Logo Jersey- M,CL-JE,Clothing,Jerseys,No,32,S,2011-07-01,2007-12-28
